# 05 – B2: Qwen2.5 0.5B / 1.5B as the scorer (Kaggle)

Baseline B2: the **same five typed questions** (`gate_questions_v2.json`) answered by an instruction-tuned Qwen instead of d1. The model sees the options as letters A, B, C…; probabilities are the softmax over the option-letter logits at the first answer token (renormalised over the options). Output has the same `probs_*` schema as the d1 results, so `scripts/analyze_gate.py` and `scripts/analyze_baselines.py` run on it unchanged. Models: `Qwen/Qwen2.5-0.5B-Instruct`, `Qwen/Qwen2.5-1.5B-Instruct` (Qwen2.5 is a public instruction model; confirm this is the intended family).

Known limitations: fixed option order (letter-position bias is possible); fp16 on T4 — if the run raises the NaN error, change `dtype=torch.float16` to `torch.float32` in `run_model`.
Kaggle: GPU on, Internet on. First cell: `!pip install -q -U transformers accelerate`.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import glob, json, os, time
import pandas as pd

xlsx = glob.glob("/kaggle/input/**/AISIS_DevSet_v1.xlsx", recursive=True)[0]
qjson = glob.glob("/kaggle/input/**/gate_questions_v2.json", recursive=True)[0]
dev = pd.read_excel(xlsx, sheet_name="DevSet").fillna("")
QUESTIONS = json.load(open(qjson, encoding="utf-8"))["questions"]
TOOLS = {k: v for k, v in QUESTIONS["tool"]["criteria"].items() if k != "none"}

# identical instance construction to notebook 01 (d1 gate): standard + dialect (+ wrong_passage for answerable)
instances = []
for r in dev.to_dict("records"):
    base = dict(dev_id=r["dev_id"], type=r["type"], topic=r["topic"],
                gold_action=r["label_action"], gold_supported=r["label_supported"],
                gold_specific=r["label_specific"], gold_emergency=r["label_emergency"],
                gold_tool=r["label_tool"])
    instances.append({**base, "version": "standard", "query": r["query_standard"], "passage": r["passage_text"]})
    instances.append({**base, "version": "dialect", "query": r["query_dialect_synth"], "passage": r["passage_text"]})
    if r["type"] == "answerable":
        instances.append({**base, "version": "wrong_passage", "query": r["query_standard"],
                          "passage": r["wrong_passage_text"],
                          "gold_action": "cannot_answer", "gold_supported": "not_supported"})
print(len(instances), "instances")  # expected 960

In [ ]:
import gc, torch, string
from transformers import AutoTokenizer, AutoModelForCausalLM

SYSTEM = ("You are a careful assistant for rural Bangladeshi users. You will be given a user query "
          "(Bengali, possibly dialectal), a retrieved passage and a multiple-choice question. "
          "Answer with the single letter of the best option and nothing else.")

def build_prompt(inst, qname):
    q = QUESTIONS[qname]
    keys = list(q["criteria"].keys())
    letters = string.ascii_uppercase[:len(keys)]
    opts = "\n".join(f"{l}. {k}: {q['criteria'][k]}" for l, k in zip(letters, keys))
    tools = "\n".join(f"- {k}: {v}" for k, v in TOOLS.items())
    user = (f"User query: {inst['query']}\n\nRetrieved passage: {inst['passage']}\n\n"
            f"Available tools:\n{tools}\n\nQuestion: {q['instructions']}\n\nOptions:\n{opts}\n\n"
            "Answer with only the letter.")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}], keys, letters

def letter_probs(tok, model, messages, keys, letters):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits[0, -1].float()
    ids = [tok.encode(l, add_special_tokens=False)[0] for l in letters]
    p = torch.softmax(logits[ids], dim=-1).cpu().tolist()   # renormalised over the option letters only
    return dict(zip(keys, p))

def run_model(name, out_csv):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda").eval()
    rows, t0 = [], time.time()
    for k, inst in enumerate(instances):
        row = {x: inst[x] for x in ["dev_id", "type", "topic", "version", "gold_action", "gold_supported",
                                    "gold_specific", "gold_emergency", "gold_tool"]}
        for qname in QUESTIONS:
            msgs, keys, letters = build_prompt(inst, qname)
            p = letter_probs(tok, model, msgs, keys, letters)
            if any(v != v for v in p.values()):
                raise RuntimeError(f"NaN at {k}/{qname}: reload in float32")   # see note in the markdown cell
            best = max(p, key=p.get)
            row[f"pred_{qname}"], row[f"conf_{qname}"] = best, p[best]
            row[f"probs_{qname}"] = json.dumps(p, ensure_ascii=False)
        rows.append(row)
        if k == 0:
            print(msgs[1]["content"][:900]); print(row)
    el = time.time() - t0
    print(f"{name}: {el:.0f}s, {1000*el/len(instances):.0f} ms/instance (5 questions), peak GB {torch.cuda.max_memory_allocated()/1e9:.2f}")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
run_model("Qwen/Qwen2.5-0.5B-Instruct", "/kaggle/working/results_qwen0.5b.csv")
run_model("Qwen/Qwen2.5-1.5B-Instruct", "/kaggle/working/results_qwen1.5b.csv")